*Importando o Pandas*

In [23]:
import pandas as pd
import numpy as np

#definindo quantidade de linhas e colunas visiveis
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

*Importação de arquivos EXCEL*

In [24]:
caminho = ('G:/Drives compartilhados/Qualidade/ACURÁCIA/DISCREPÂNCIA/DISCREPÂNCIA DIAGNÓSTICA/Discrepância 2026/Mar/')
df = pd.read_excel(caminho + 'Discrepancia Osasco Mar 2026.xlsx')
df_bck = df.copy()

In [ ]:
df['Ds setor atendimento'].value_counts()

*Criando o dataframe de CIDs.*

In [ ]:
df.loc[df['Validacao cid'] == "CID DIFERENTE", 'Cor'] = ""

*Remover linhas que não contem resposta*

In [ ]:
# O raciocínio: Localize as linhas onde isnull() é True e mude a coluna 'Cor'
df.loc[df['Diagnostico pós'].isna() | (df['Diagnostico pós'] == ''), 'Cor'] = "Amarelo"
print(df.loc[df['Diagnostico pós'].isnull()])

*Removendo CIDs*

In [ ]:
import numpy as np

# 1. Manter a limpeza de nulos inicial
df['Diagnostico pre'] = df['Diagnostico pre'].fillna('')

# 2. Identificação e Atribuição da Cor
padrao_cid = r"[A-Z]\d+(?:\.\d+)?"
df['Cor'] = np.where(
    df['Diagnostico pre'].str.contains(padrao_cid, regex=True, na=False), 
    'Roxo', 
    ''
)

# 3. A "Lanterna" (Máscara Booleana)
# Criamos um filtro para identificar onde a Cor está vazia
mask_vazio = df['Cor'] == ''

# 4. Aplicação Cirúrgica
# Aplicamos as transformações de string APENAS nas linhas da máscara
df.loc[mask_vazio, 'Diagnostico pre'] = (
    df.loc[mask_vazio, 'Diagnostico pre']
    .str.replace("_x000D_", " ", regex=False)
    .str.strip()
)

print(df[['Diagnostico pre', 'Cor']].head())

In [ ]:
import numpy as np

# 1. Manter a limpeza de nulos inicial
df['Diagnostico pós'] = df['Diagnostico pós'].fillna('')

# 2. Identificação e Atribuição da Cor
padrao_cid = r"[A-Z]\d+(?:\.\d+)?"
df['Cor'] = np.where(
    df['Diagnostico pós'].str.contains(padrao_cid, regex=True, na=False), 
    'Roxo', 
    ''
)

# 3. A "Lanterna" (Máscara Booleana)
# Criamos um filtro para identificar onde a Cor está vazia
mask_vazio = df['Cor'] == ''

# 4. Aplicação Cirúrgica
# Aplicamos as transformações de string APENAS nas linhas da máscara
df.loc[mask_vazio, 'Diagnostico pós'] = (
    df.loc[mask_vazio, 'Diagnostico pós']
    .str.replace("_x000D_", " ", regex=False)
    .str.strip()
)

print(df[['Diagnostico pós', 'Cor']].head())

In [ ]:
df.loc[df['Validacao cid']== "CID IGUAL", 'Cor'] = 'Verde'

*Classificação de Dados*

In [ ]:
df.sort_values(by=["Nome paciente"])

In [ ]:
contagem = df['Ds setor atendimento'].value_counts()
print(contagem)

*Filtrando os dados*

In [ ]:
# Filtrando linhas coma palavra Convalescenca
conva = df['Diagnostico pós'].str.contains('CONVALESCENCA', na=False)
df.loc[conva, 'Cor'] = "Amarelo"
# Exibindo as linhas corretas
print(df[df['Diagnostico pós'].str.contains('CONVALESCENCA', na=False)])

In [ ]:
outros = df['Diagnostico pós'].str.contains('OUTROS', na=False)
df.loc[outros, 'Cor'] = " Amarelo"

print(df[df['Diagnostico pós'].str.contains('OUTROS', na=False)])

In [ ]:
segui = df['Diagnostico pós'].str.contains('Seguimentos', na=False)
df.loc[segui, 'Cor'] = " Amarelo"

print(df[df['Diagnostico pós'].str.contains('Seguimentos', na=False)])

*Verificando score entre palavras*

In [ ]:
from thefuzz import process
from thefuzz import fuzz

df['Diagnostico pós'] = df['Diagnostico pós'].fillna('')

df['score_fuzz.ratio'] = 0
df['score_fuzz.ratio'] = df['score_fuzz.ratio'].astype('float')
df['score_fuzz.token_sort_ratio'] = df['score_fuzz.ratio']
df['score_fuzz.partial_ratio'] = df['score_fuzz.ratio']

for linha in df.index:
    
    texto1 = df.loc[linha, 'Diagnostico pre']
    texto2 = df.loc[linha, 'Diagnostico pós']
    # ===================================================
    melhor, score = process.extractOne(
        query=texto1,
        choices=[texto2],
        scorer=fuzz.ratio
    )
    df.loc[linha, 'score_fuzz.ratio'] = score
    # ===================================================
    melhor, score = process.extractOne(
        query=texto1,
        choices=[texto2],
        scorer=fuzz.token_sort_ratio
    )
    df.loc[linha, 'score_fuzz.token_sort_ratio'] = score
    # ===================================================
    melhor, score = process.extractOne(
        query=texto1,
        choices=[texto2],
        scorer=fuzz.partial_ratio
    )
    df.loc[linha, 'score_fuzz.partial_ratio'] = score
    # ===================================================
    #display(df.loc[linha:linha, ['Diagnostico pre','Diagnostico pós','score']])

In [ ]:
df_bck.loc[140, 'Diagnostico pre'][2]

In [ ]:
display(df_bck.loc[57:57])

display(df.loc[57:57])

In [ ]:
display(df[['Diagnostico pre','Diagnostico pós','score_fuzz.ratio','score_fuzz.token_sort_ratio','score_fuzz.partial_ratio']].sample(20))

*Retirar linhas com score acima de 70*

In [ ]:
roxo = (
        (df['score_fuzz.ratio'] > 50) &
        (df['score_fuzz.token_sort_ratio'] > 50) &
        (df['score_fuzz.partial_ratio'] > 50))

condicao_vazio = (df['Cor'].isna()) | (df['Cor'] == '')

df.loc[roxo & condicao_vazio, 'Cor'] = 'Roxo'

In [ ]:
df.to_excel("Discrepancia Osasco Mar 2026(teste1).xlsx")